In [ ]:
# %pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
# %pip install numpy pandas pillow scikit-learn matplotlib tqdm

In [12]:
import os, sys, json, hashlib, random, time, math, unicodedata
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from IPython.display import display

.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
DATA = Path('data').resolve()
if not DATA.is_dir():
    DATA = Path(r'data')
assert DATA.is_dir(), f'Ajusta DATA: {DATA}'
OUT = DATA.parent / "kexia_v4_runs" / "20261124_102050_038611"

assert OUT.is_dir(), "No existe la carpeta original"
SEED=42
CLASSES=['angry','disgust','fear','happy','neutral','sad','surprise']
CLASS_ID={c:i for i,c in enumerate(CLASSES)}
BATCH=8; ACCUM=4; WORKERS=0  # Windows/Jupyter
CNN_SIZE=160; VIT_SIZE=224; VAL_FRACTION=.15
FUSION='gated'  # 'concat' también disponible
MODE='hybrid'  # 'cnn' o 'vit' para ablaciones con el mismo manifiesto
RUN_TRAIN=False; RUN_TEST=False; ALLOW_CPU=False
PRETRAINED=False
# Etiquetas numéricas solo si su significado está documentado: {'0':'angry', ...}
LABEL_MAP={}
# Reutilizar un manifiesto auditado (por ejemplo para ablaciones):
REUSE_MANIFEST = Path(r"kexia_v4_runs\20261124_102050_038611\manifest.csv")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:',sys.executable,'Torch:',torch.__version__,'CUDA:',torch.version.cuda,'device:',device)
if device.type=='cuda':
    print(torch.cuda.get_device_name(0), 'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
    print('Prueba CUDA:',(torch.ones(8,device=device)*2).sum().item())
print('DATA:',DATA,'OUT:',OUT)

Python: .venv\Scripts\python.exe Torch: 2.8.0+cu128 CUDA: 12.8 device: cuda
NVIDIA GeForce RTX 5070 Ti Laptop GPU VRAM GiB: 11.94
Prueba CUDA: 16.0
DATA: data OUT: kexia_v4_runs\20261124_102050_038611


In [15]:
def norm(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',s.casefold()) if c.isalnum() and not unicodedata.combining(c))
ALIASES={}
for cls, names in {
 'angry':['anger','angry','enojo','ira'], 'disgust':['disgust','disgusted','asco','disgusto'],
 'fear':['fear','fearful','miedo'], 'happy':['happy','happiness','felicidad','alegria'],
 'neutral':['neutral','neutrality','neutro'], 'sad':['sad','sadness','tristeza'],
 'surprise':['surprise','surprised','sorpresa']}.items():
    ALIASES.update({norm(x):cls for x in names})
SPLITS={norm(x):s for s,xs in {'train':['train','training'],'validation':['valid','val','validation','dev'],'test':['test','testing']}.items() for x in xs}
EXT={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
def parse_path(p):
    parts=p.relative_to(DATA).parts
    if len(parts)<3: return None,'unknown_layout'
    split=SPLITS.get(norm(parts[0]))
    if split is None: return None,'unknown_split'
    original=parts[1]
    mapping={norm(k):v for k,v in LABEL_MAP.items()}
    cls=mapping.get(norm(original),ALIASES.get(norm(original)))
    if cls not in CLASS_ID: return None,'unknown_label'
    return (split,original,cls),None
def read_image(p):
    with Image.open(p) as im: return ImageOps.exif_transpose(im).convert('RGB')
def hashes(p,im):
    a=np.asarray(im); h=hashlib.sha256(str(a.shape).encode()+a.tobytes()).hexdigest()
    g=np.asarray(im.convert('L').resize((9,8),Image.Resampling.LANCZOS))
    dh=np.packbits((g[:,1:]>g[:,:-1]).ravel()).tobytes().hex()
    return hashlib.sha256(p.read_bytes()).hexdigest(),h,dh
if REUSE_MANIFEST is None:
    rows=[]; issues=[]
    roots=[p for p in DATA.iterdir() if p.is_dir() and norm(p.name) in SPLITS]
    split_names=[SPLITS[norm(p.name)] for p in roots]
    assert len(split_names)==len(set(split_names)), 'Hay varias carpetas para una misma partición; revisa DATA.'
    assert {'train','validation'}.issubset(split_names), 'Se requiere data/train y data/valid (o val/validation).'
    files=sorted(p for root in roots for p in root.rglob('*') if p.is_file() and p.suffix.lower() in EXT)
    for p in tqdm(files,desc='Auditoría de imágenes'):
        parsed,err=parse_path(p)
        if err: issues.append({'path':str(p),'reason':err}); continue
        split,original,cls=parsed
        try:
            im=read_image(p); filehash,pixelhash,dh=hashes(p,im)
            rows.append(dict(path=str(p),original_split=split,split=split,original_label=original,cls=cls,label=CLASS_ID[cls],file_hash=filehash,pixel_hash=pixelhash,dhash=dh,width=im.width,height=im.height))
        except Exception as e: issues.append({'path':str(p),'reason':str(e)})
    pd.DataFrame(issues,columns=['path','reason']).to_csv(OUT/'scan_issues.csv',index=False)
    assert rows,'No se encontraron imágenes reconocidas; revisa scan_issues.csv y configuración.'
    raw=pd.DataFrame(rows); display(pd.crosstab(raw.original_split,raw.cls))
    conflict=raw.groupby('pixel_hash').label.nunique(); bad=set(conflict[conflict>1].index)
    priority={'test':0,'validation':1,'train':2}
    clean=raw[~raw.pixel_hash.isin(bad)].copy()
    clean['priority']=clean.split.map(priority)
    clean=clean.sort_values(['priority','pixel_hash','path']).drop_duplicates('pixel_hash').drop(columns='priority')
    excluded=raw[~raw.path.isin(clean.path)].copy()
    excluded['reason']=np.where(excluded.pixel_hash.isin(bad),'conflicting_label','exact_duplicate')
    excluded.to_csv(OUT/'excluded_images.csv',index=False)
    data=clean.reset_index(drop=True)
    audit={'scanned':len(files),'decoded':len(raw),'issues':len(issues),'excluded':len(excluded),'conflicting_hashes':len(bad),'cross_split_hashes':int((raw.groupby('pixel_hash').split.nunique()>1).sum()),'retained':len(data)}
    (OUT/'dataset_audit.json').write_text(json.dumps(audit,indent=2)); print(audit)
else:
    data=pd.read_csv(REUSE_MANIFEST,dtype={'pixel_hash':str,'dhash':str,'file_hash':str})
assert data.pixel_hash.is_unique
assert all(data.label==data.cls.map(CLASS_ID))
active_splits=['train','validation']+(['test'] if 'test' in set(data.split) else [])
counts=pd.crosstab(data.cls,data.split).reindex(index=CLASSES,columns=active_splits,fill_value=0)
display(counts)
assert (counts>0).all().all(), 'Faltan clases en alguna partición tras auditar; revisa distribution y exclusiones.'
if 'test' not in active_splits: print('Sin test independiente: solo se reportará validación.')
data.to_csv(OUT/'manifest.csv',index=False); counts.to_csv(OUT/'distribution.csv')
(OUT/'labels.json').write_text(json.dumps(CLASS_ID,indent=2))
# dHash repetidos entre particiones: candidatos, no duplicados confirmados.
near=data.groupby('dhash').split.nunique(); candidates=data[data.dhash.isin(near[near>1].index)]
candidates.to_csv(OUT/'perceptual_candidates.csv',index=False)
print('Candidatos dHash cruzados (requieren revisión):',len(candidates))

split,train,validation,test
cls,,,
angry,10046,1002,1975
disgust,3301,651,756
fear,8325,647,1688
happy,34168,5308,6973
neutral,33147,5772,7052
sad,16619,2208,3402
surprise,9682,1429,2244


Candidatos dHash cruzados (requieren revisión): 4865


In [16]:
augment=T.Compose([T.RandomHorizontalFlip(),T.RandomAffine(12,translate=(.05,.05),scale=(.92,1.08)),T.ColorJitter(.15,.15,.08)])
mean=(.485,.456,.406); std=(.229,.224,.225)
def transform(size): return T.Compose([T.Resize((size,size),antialias=True),T.ToTensor(),T.Normalize(mean,std)])
cnn_tf=transform(CNN_SIZE); vit_tf=transform(VIT_SIZE)
class FERDataset(Dataset):
    def __init__(self,frame,training=False): self.frame=frame.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; im=read_image(Path(r.path))
        if self.training: im=augment(im)
        return cnn_tf(im),vit_tf(im),int(r.label),i
sets={s:FERDataset(data[data.split==s],s=='train') for s in active_splits}
def loader(s):
    return DataLoader(sets[s],batch_size=BATCH,shuffle=s=='train',num_workers=WORKERS,pin_memory=device.type=='cuda',generator=torch.Generator().manual_seed(SEED))
loaders={s:loader(s) for s in sets}
weights=np.sqrt(counts.train.sum()/(7*counts.train.to_numpy(float)))
weights/=np.average(weights,weights=counts.train)
criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=device))
print('Pesos:',dict(zip(CLASSES,weights.round(3))))

Pesos: {'angry': 1.364, 'disgust': 2.38, 'fear': 1.499, 'happy': 0.74, 'neutral': 0.751, 'sad': 1.061, 'surprise': 1.39}


In [17]:
class HybridFER(nn.Module):
    def __init__(self,pretrained=True,mode='hybrid',fusion='gated'):
        super().__init__(); self.mode=mode; self.fusion=fusion
        assert mode in ('cnn','vit','hybrid') and fusion in ('gated','concat')
        self.cnn=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None) if mode!='vit' else None
        self.vit=models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT if pretrained else None) if mode!='cnn' else None
        if self.cnn is not None: self.cnn.classifier=nn.Identity()
        if self.vit is not None: self.vit.heads=nn.Identity()
        def project(d): return nn.Sequential(nn.LayerNorm(d),nn.Linear(d,512),nn.GELU(),nn.Dropout(.25))
        self.pc=project(1280) if self.cnn is not None else None
        self.pv=project(768) if self.vit is not None else None
        self.gate=nn.Sequential(nn.Linear(1024,512),nn.Sigmoid()) if mode=='hybrid' and fusion=='gated' else None
        d=1024 if mode=='hybrid' and fusion=='concat' else 512
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,256),nn.GELU(),nn.Dropout(.4),nn.Linear(256,7))
    def forward(self,x,y,return_gate=False):
        c=self.pc(self.cnn(x)) if self.cnn is not None else None
        v=self.pv(self.vit(y)) if self.vit is not None else None
        g=None
        if self.mode=='hybrid':
            z=torch.cat([c,v],1)
            if self.gate is not None: g=self.gate(z); z=g*c+(1-g)*v
        else: z=c if c is not None else v
        logits=self.head(z)
        return (logits,g) if return_gate else logits
model=HybridFER(PRETRAINED,MODE,FUSION).to(device)
print('Parámetros totales:',sum(p.numel() for p in model.parameters()))
def configure_stage(stage):
    for p in model.parameters(): p.requires_grad_(True)
    for b in [model.cnn,model.vit]:
        if b is not None:
            for p in b.parameters(): p.requires_grad_(False)
    if stage!='A':
        if model.cnn is not None:
            for block in list(model.cnn.features)[-(4 if stage=='B' else 8):]:
                for p in block.parameters(): p.requires_grad_(True)
        if model.vit is not None:
            for block in list(model.vit.encoder.layers)[-(2 if stage=='B' else 4):]:
                for p in block.parameters(): p.requires_grad_(True)
            for p in model.vit.encoder.ln.parameters(): p.requires_grad_(True)
    if model.cnn is not None:
        for m in model.cnn.modules():
            if isinstance(m,nn.BatchNorm2d):
                for p in m.parameters(): p.requires_grad_(False)
def train_mode():
    model.train()
    for b in [model.cnn,model.vit]:
        if b is not None and not any(p.requires_grad for p in b.parameters()): b.eval()
    for m in model.modules():
        if isinstance(m,nn.BatchNorm2d): m.eval()
configure_stage('A'); train_mode()
x,y,t,_=next(iter(loaders['train']))
x,y,t=x[:2].to(device),y[:2].to(device),t[:2].to(device)
logits=model(x,y); assert logits.shape==(len(t),7)
loss=criterion(logits,t); loss.backward(); assert model.head[-1].weight.grad is not None
model.zero_grad(set_to_none=True); print('Smoke test:',logits.shape,'loss:',loss.item())
del x,y,t,logits,loss

Parámetros totales: 89735175
Smoke test: torch.Size([2, 7]) loss: 1.9707282781600952


In [18]:
RUN_TRAIN=False  # Cambiar a True para entrenar
STAGES=[('A',8,4,1e-3,0,0),('B',20,6,1e-4,3e-5,1e-5),('C',25,7,5e-5,1e-5,5e-6)]
amp=device.type=='cuda'
amp_dtype=torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16
scaler=torch.amp.GradScaler('cuda',enabled=amp and amp_dtype==torch.float16)
@torch.no_grad()
def evaluate(split):
    model.eval(); ys=[]; probs=[]; ids=[]; gates=[]; total=0; count=0
    for x,y,t,i in tqdm(loaders[split],desc=split,leave=False):
        x,y,t=x.to(device),y.to(device),t.to(device)
        with torch.autocast(device_type=device.type,dtype=amp_dtype,enabled=amp): logits,g=model(x,y,True)
        total+=nn.functional.cross_entropy(logits.float(),t,reduction='sum').item(); count+=len(t)
        ys.extend(t.cpu().tolist()); probs.extend(logits.float().softmax(1).cpu().numpy()); ids.extend(i.tolist())
        if g is not None: gates.extend(g.float().mean(1).cpu().tolist())
    ys=np.array(ys); probs=np.array(probs); pred=probs.argmax(1)
    metrics={'loss':total/count,'accuracy':accuracy_score(ys,pred),'balanced_accuracy':balanced_accuracy_score(ys,pred),'macro_f1':f1_score(ys,pred,labels=range(7),average='macro',zero_division=0),'weighted_f1':f1_score(ys,pred,average='weighted',zero_division=0)}
    if gates: metrics['mean_cnn_gate']=float(np.mean(gates))
    return metrics,ys,pred,probs,ids,gates
BEST=OUT/'best.pt'; history=[]; global_best=-1
config=dict(mode=MODE,fusion=FUSION,cnn_size=CNN_SIZE,vit_size=VIT_SIZE,classes=CLASSES,seed=SEED,batch=BATCH,accum=ACCUM,pretrained=PRETRAINED)
(OUT/'config.json').write_text(json.dumps(config,indent=2))
if RUN_TRAIN:
    assert amp or ALLOW_CPU,'CUDA no disponible. Corrige PyTorch o habilita ALLOW_CPU explícitamente.'
    for stage,epochs,patience,head_lr,cnn_lr,vit_lr in STAGES:
        configure_stage(stage)
        groups=[]
        for name,lr in [('cnn',cnn_lr),('vit',vit_lr),('head',head_lr)]:
            ps=[p for k,p in model.named_parameters() if p.requires_grad and ((k.startswith('cnn.') if name=='cnn' else k.startswith('vit.') if name=='vit' else not k.startswith(('cnn.','vit.'))))]
            if ps: groups.append({'params':ps,'lr':lr})
        opt=torch.optim.AdamW(groups,weight_decay=.01)
        steps=math.ceil(len(loaders['train'])/ACCUM); total_steps=steps*epochs; warm=max(1,int(.05*total_steps))
        def factor(s): return min(1,(s+1)/warm) if s<warm else .05+.95*(1+math.cos(math.pi*min(1,(s-warm)/max(1,total_steps-warm))))/2
        sched=torch.optim.lr_scheduler.LambdaLR(opt,factor)
        stage_best=-1; stale=0
        print('Etapa',stage,'entrenables:',sum(p.numel() for p in model.parameters() if p.requires_grad))
        for ep in range(1,epochs+1):
            train_mode(); opt.zero_grad(set_to_none=True); start=time.perf_counter(); trloss=0.; correct=0; seen=0
            if amp: torch.cuda.reset_peak_memory_stats()
            for j,(x,y,t,_) in enumerate(tqdm(loaders['train'],desc=f'{stage} {ep}/{epochs}',leave=False)):
                x,y,t=x.to(device),y.to(device),t.to(device)
                group_len=min(ACCUM,len(loaders['train'])-(j//ACCUM)*ACCUM)
                with torch.autocast(device_type=device.type,dtype=amp_dtype,enabled=amp):
                    logits=model(x,y); loss=criterion(logits,t)
                scaler.scale(loss/group_len).backward()
                if (j+1)%ACCUM==0 or j+1==len(loaders['train']):
                    scaler.unscale_(opt); nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.)
                    before=scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                    if scaler.get_scale()>=before: sched.step()
                seen+=len(t); trloss+=loss.item()*len(t); correct+=(logits.argmax(1)==t).sum().item()
            val,*_=evaluate('validation'); score=val['macro_f1']
            row=dict(stage=stage,epoch=ep,train_loss=trloss/seen,train_accuracy=correct/seen,seconds=time.perf_counter()-start,lr=opt.param_groups[-1]['lr'],**{'val_'+k:v for k,v in val.items()})
            if amp: row['peak_gpu_gib']=torch.cuda.max_memory_allocated()/2**30
            history.append(row); pd.DataFrame(history).to_csv(OUT/'metrics_history.csv',index=False); print(row)
            if score>global_best:
                global_best=score; torch.save({'state_dict':model.state_dict(),'config':config,'val':val,'stage':stage,'epoch':ep},BEST)
            if score>stage_best+1e-5: stage_best=score; stale=0
            else: stale+=1
            if stale>=patience: print('Early stopping',stage); break
        model.load_state_dict(torch.load(BEST,map_location=device,weights_only=True)['state_dict'])
    print('Mejor Macro F1:',global_best,'Checkpoint:',BEST)
else: print('Entrenamiento desactivado. Cambia RUN_TRAIN=True en esta celda.')

Entrenamiento desactivado. Cambia RUN_TRAIN=True en esta celda.


In [19]:
from pathlib import Path

ROOT = Path(r"kexia_v4_runs")

for p in ROOT.rglob("best.pt"):
    print(p)

kexia_v4_runs\20261124_102050_038611\best.pt


In [20]:
from pathlib import Path
import torch

# Detectar GPU automáticamente
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BEST = Path(r"kexia_v4_runs\20261124_102050_038611\best.pt")

OUT = BEST.parent

assert BEST.exists(), "No se encontró el checkpoint"

checkpoint = torch.load(
    BEST,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint recuperado correctamente")
print("Etapa:", checkpoint["stage"])
print("Época:", checkpoint["epoch"])
print("Macro F1:", checkpoint["val"]["macro_f1"])
print("Accuracy:", checkpoint["val"]["accuracy"])
print("Dispositivo:", device)

if "model" in globals():
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    print("Pesos cargados correctamente")
else:
    print("Falta inicializar la arquitectura del modelo")

print("Carpeta:", OUT)

Checkpoint recuperado correctamente
Etapa: B
Época: 5
Macro F1: 0.40676551538629735
Accuracy: 0.5324087677028854
Dispositivo: cuda
Pesos cargados correctamente
Carpeta: kexia_v4_runs\20261124_102050_038611


In [21]:

import torch
import math
import time
import shutil
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURACIÓN DE RECUPERACIÓN
# ==========================================

BEST = Path(r"kexia_v4_runs\20261124_102050_038611\best.pt")

OUT = BEST.parent

BACKUP = OUT / "best_original.pt"

if not BACKUP.exists():
    shutil.copy2(BEST, BACKUP)
    print("Respaldo original creado correctamente")

LAST = OUT / "last.pt"
HISTORY = OUT / "metrics_history.csv"
RESUME_HISTORY = OUT / "metrics_resumed.csv"

assert BEST.exists()
assert "model" in globals(), "Inicializa la arquitectura"
assert "loaders" in globals(), "Inicializa los DataLoaders"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "GPU CUDA no disponible"

# Recuperar último estado disponible
source = LAST if LAST.exists() else BEST

checkpoint = torch.load(
    source,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(checkpoint["state_dict"])
model.to(device)

config = checkpoint["config"]

resume_stage = checkpoint["stage"]
resume_epoch = checkpoint["epoch"]

best_checkpoint = torch.load(
    BEST,
    map_location="cpu",
    weights_only=True
)

global_best = best_checkpoint["val"]["macro_f1"]

del best_checkpoint

# Conservar las métricas anteriores
if RESUME_HISTORY.exists():
    history = pd.read_csv(
        RESUME_HISTORY
    ).to_dict("records")
elif HISTORY.exists():
    history = pd.read_csv(
        HISTORY
    ).to_dict("records")
else:
    history = []

# Identificar la nueva rama del entrenamiento
if source == BEST:
    history = [
        r for r in history
        if r["stage"] < resume_stage
        or (
            r["stage"] == resume_stage
            and int(r["epoch"]) <= resume_epoch
        )
    ]

STAGES = [
    ("B", 20, 6, 1e-4, 3e-5, 1e-5),
    ("C", 25, 7, 5e-5, 1e-5, 5e-6)
]

amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_dtype == torch.float16
)

# ==========================================
# GUARDADO ATÓMICO
# ==========================================

def save_checkpoint(state, path):
    temp = path.with_suffix(".tmp")
    torch.save(state, temp)
    temp.replace(path)


# ==========================================
# EVALUACIÓN
# ==========================================

@torch.no_grad()
def evaluate_resume(split):
    model.eval()

    ys = []
    probs = []
    total = 0
    count = 0

    for x, y, t, i in tqdm(
        loaders[split],
        desc=split,
        leave=False
    ):
        x = x.to(device)
        y = y.to(device)
        t = t.to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp
        ):
            logits = model(x, y)

        total += nn.functional.cross_entropy(
            logits.float(),
            t,
            reduction="sum"
        ).item()

        count += len(t)
        ys.extend(t.cpu().tolist())
        probs.extend(
            logits.float().softmax(1).cpu().numpy()
        )

    ys = np.array(ys)
    probs = np.array(probs)
    pred = probs.argmax(1)

    return {
        "loss": total / count,
        "accuracy": accuracy_score(ys, pred),
        "balanced_accuracy": balanced_accuracy_score(
            ys, pred
        ),
        "macro_f1": f1_score(
            ys, pred,
            labels=range(7),
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            ys, pred,
            average="weighted",
            zero_division=0
        )
    }


# ==========================================
# CONTINUACIÓN DEL ENTRENAMIENTO
# ==========================================

print("Checkpoint:", source)
print("Etapa recuperada:", resume_stage)
print("Época recuperada:", resume_epoch)
print("Mejor Macro F1:", global_best)
print("GPU:", torch.cuda.get_device_name(0))

for stage, epochs, patience, head_lr, cnn_lr, vit_lr in STAGES:

    if stage < resume_stage:
        continue

    configure_stage(stage)

    groups = []

    for name, lr in [
        ("cnn", cnn_lr),
        ("vit", vit_lr),
        ("head", head_lr)
    ]:

        ps = [
            p for k, p in model.named_parameters()
            if p.requires_grad and (
                k.startswith("cnn.") if name == "cnn"
                else k.startswith("vit.") if name == "vit"
                else not k.startswith(("cnn.", "vit."))
            )
        ]

        if ps:
            groups.append({
                "params": ps,
                "lr": lr
            })

    opt = torch.optim.AdamW(
        groups,
        weight_decay=.01
    )

    start_epoch = (
        resume_epoch + 1
        if stage == resume_stage
        else 1
    )

    steps = math.ceil(
        len(loaders["train"]) / ACCUM
    )

    total_steps = steps * epochs
    warm = max(1, int(.05 * total_steps))

    def factor(s):
        if s < warm:
            return min(1, (s + 1) / warm)

        return .05 + .95 * (
            1 + math.cos(
                math.pi * min(
                    1,
                    (s - warm) /
                    max(1, total_steps - warm)
                )
            )
        ) / 2

    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        factor
    )

    stage_best = (
        checkpoint["val"]["macro_f1"]
        if stage == resume_stage
        else -1
    )

    stale = 0

    # Recuperar optimizador y scheduler si existen
    if stage == resume_stage:

        if "optimizer" in checkpoint:
            opt.load_state_dict(
                checkpoint["optimizer"]
            )

            sched.load_state_dict(
                checkpoint["scheduler"]
            )

            scaler.load_state_dict(
                checkpoint["scaler"]
            )

            stage_best = checkpoint.get(
                "stage_best", stage_best
            )

            stale = checkpoint.get("stale", 0)

        else:
            # Aproximar la posición original del scheduler
            completed_steps = steps * resume_epoch

            sched.last_epoch = completed_steps

            for group, base_lr in zip(
                opt.param_groups,
                sched.base_lrs
            ):
                group["lr"] = (
                    base_lr * factor(completed_steps)
                )

    if stale >= patience:
        model.load_state_dict(
            torch.load(
                BEST,
                map_location=device,
                weights_only=True
            )["state_dict"]
        )
        continue

    print("\n================================")
    print("ETAPA:", stage)
    print("Época inicial:", start_epoch)
    print("Época máxima:", epochs)
    print("================================")

    for ep in range(start_epoch, epochs + 1):

        train_mode()
        opt.zero_grad(set_to_none=True)

        start = time.perf_counter()
        trloss = 0.
        correct = 0
        seen = 0

        torch.cuda.reset_peak_memory_stats()

        for j, (x, y, t, _) in enumerate(
            tqdm(
                loaders["train"],
                desc=f"{stage} {ep}/{epochs}"
            )
        ):

            x = x.to(device)
            y = y.to(device)
            t = t.to(device)

            group_len = min(
                ACCUM,
                len(loaders["train"])
                - (j // ACCUM) * ACCUM
            )

            with torch.autocast(
                device_type=device.type,
                dtype=amp_dtype,
                enabled=amp
            ):
                logits = model(x, y)
                loss = criterion(logits, t)

            scaler.scale(
                loss / group_len
            ).backward()

            if (
                (j + 1) % ACCUM == 0
                or j + 1 == len(loaders["train"])
            ):

                scaler.unscale_(opt)

                nn.utils.clip_grad_norm_(
                    [
                        p for p in model.parameters()
                        if p.requires_grad
                    ],
                    1.
                )

                before = scaler.get_scale()

                scaler.step(opt)
                scaler.update()

                opt.zero_grad(set_to_none=True)

                if scaler.get_scale() >= before:
                    sched.step()

            seen += len(t)
            trloss += loss.item() * len(t)
            correct += (
                logits.argmax(1) == t
            ).sum().item()

        val = evaluate_resume("validation")
        score = val["macro_f1"]

        row = dict(
            stage=stage,
            epoch=ep,
            train_loss=trloss / seen,
            train_accuracy=correct / seen,
            seconds=time.perf_counter() - start,
            lr=opt.param_groups[-1]["lr"],
            peak_gpu_gib=(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            **{
                "val_" + k: v
                for k, v in val.items()
            }
        )

        history.append(row)

        pd.DataFrame(history).to_csv(
            RESUME_HISTORY,
            index=False
        )

        if score > stage_best + 1e-5:
            stage_best = score
            stale = 0
        else:
            stale += 1

        # Guardar el mejor modelo
        if score > global_best:

            global_best = score

            save_checkpoint({
                "state_dict": model.state_dict(),
                "config": config,
                "val": val,
                "stage": stage,
                "epoch": ep
            }, BEST)

            print("Nuevo mejor modelo guardado")

        # Guardar el estado completo de recuperación
        save_checkpoint({
            "state_dict": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(),
            "scaler": scaler.state_dict(),
            "config": config,
            "val": val,
            "stage": stage,
            "epoch": ep,
            "global_best": global_best,
            "stage_best": stage_best,
            "stale": stale
        }, LAST)

        print(row)
        print("Checkpoint guardado:", LAST)

        if stale >= patience:
            print("Early stopping:", stage)
            break

    model.load_state_dict(
        torch.load(
            BEST,
            map_location=device,
            weights_only=True
        )["state_dict"]
    )

print("\nENTRENAMIENTO FINALIZADO")
print("Mejor Macro F1:", global_best)
print("Mejor modelo:", BEST)
print("Último checkpoint:", LAST)


Respaldo original creado correctamente
Checkpoint: kexia_v4_runs\20261124_102050_038611\best.pt
Etapa recuperada: B
Época recuperada: 5
Mejor Macro F1: 0.40676551538629735
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU

ETAPA: B
Época inicial: 6
Época máxima: 20


B 6/20: 100%|██████████| 14411/14411 [27:13<00:00,  8.82it/s] 


{'stage': 'B', 'epoch': 6, 'train_loss': 0.9828191743910946, 'train_accuracy': 0.6740770938866144, 'seconds': 1722.380699899979, 'lr': 8.46708746522227e-05, 'peak_gpu_gib': 1.051496982574463, 'val_loss': 1.3337425281363031, 'val_accuracy': 0.5312922371745901, 'val_balanced_accuracy': 0.4066944361919126, 'val_macro_f1': 0.401086566759623, 'val_weighted_f1': 0.5193068857775368}
Checkpoint guardado: kexia_v4_runs\20261124_102050_038611\last.pt


B 7/20: 100%|██████████| 14411/14411 [12:06<00:00, 19.83it/s]


{'stage': 'B', 'epoch': 7, 'train_loss': 0.945621483508234, 'train_accuracy': 0.6845638748178475, 'seconds': 820.5233623000095, 'lr': 7.848003751081529e-05, 'peak_gpu_gib': 1.051496982574463, 'val_loss': 1.4047733393708526, 'val_accuracy': 0.5174237527178703, 'val_balanced_accuracy': 0.41861627110677796, 'val_macro_f1': 0.40464741853828035, 'val_weighted_f1': 0.5111376753756058}
Checkpoint guardado: kexia_v4_runs\20261124_102050_038611\last.pt


B 8/20: 100%|██████████| 14411/14411 [11:59<00:00, 20.04it/s]


... (43 outputs left out)

C 11/25: 100%|██████████| 14411/14411 [2:21:08<00:00,  1.70it/s]     


{'stage': 'C', 'epoch': 11, 'train_loss': 0.6744999997326192, 'train_accuracy': 0.7697852335021859, 'seconds': 8588.188400299987, 'lr': 3.2837948726090186e-05, 'peak_gpu_gib': 1.4816927909851074, 'val_loss': 1.6562656133976443, 'val_accuracy': 0.5067873303167421, 'val_balanced_accuracy': 0.44508499159452913, 'val_macro_f1': 0.4153141334725458, 'val_weighted_f1': 0.5187911499195081}
Checkpoint guardado: kexia_v4_runs\20261124_102050_038611\last.pt


C 12/25: 100%|██████████| 14411/14411 [14:09<00:00, 16.96it/s]


{'stage': 'C', 'epoch': 12, 'train_loss': 0.661263990797137, 'train_accuracy': 0.7733762403719381, 'seconds': 940.6568885000888, 'lr': 2.9770907638508105e-05, 'peak_gpu_gib': 1.4816927909851074, 'val_loss': 1.6555814140515637, 'val_accuracy': 0.5134865134865135, 'val_balanced_accuracy': 0.43862415489239165, 'val_macro_f1': 0.4174959437137681, 'val_weighted_f1': 0.518699474566413}
Checkpoint guardado: kexia_v4_runs\20261124_102050_038611\last.pt
Early stopping: C

ENTRENAMIENTO FINALIZADO
Mejor Macro F1: 0.4255765952025515
Mejor modelo: kexia_v4_runs\20261124_102050_038611\best.pt
Último checkpoint: kexia_v4_runs\20261124_102050_038611\last.pt


In [ ]:
RUN_TEST=True  # Activar solo tras fijar el modelo
if RUN_TEST and 'test' not in loaders: print('No existe test independiente; se evaluará únicamente validación.')
CHECKPOINT=BEST  # O ruta de un checkpoint anterior compatible
if CHECKPOINT.exists():
    ck=torch.load(CHECKPOINT,map_location=device,weights_only=True)
    assert ck['config']['classes']==CLASSES and ck['config']['mode']==MODE and ck['config']['fusion']==FUSION
    model.load_state_dict(ck['state_dict'])
    for split in (['validation','test'] if RUN_TEST and 'test' in loaders else ['validation']):
        met,ys,pred,probs,ids,gates=evaluate(split); print(split,met)
        (OUT/f'{split}_metrics.json').write_text(json.dumps(met,indent=2))
        report=classification_report(ys,pred,labels=list(range(7)),target_names=CLASSES,output_dict=True,zero_division=0)
        pd.DataFrame(report).T.to_csv(OUT/f'{split}_classification_report.csv'); display(pd.DataFrame(report).T)
        frame=sets[split].frame.iloc[ids][['path','cls']].reset_index(drop=True)
        frame['predicted']=[CLASSES[i] for i in pred]
        for j,c in enumerate(CLASSES): frame['p_'+c]=probs[:,j]
        if gates: frame['cnn_gate']=gates
        frame.to_csv(OUT/f'{split}_predictions.csv',index=False)
        cm=confusion_matrix(ys,pred,labels=range(7)); pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/f'{split}_confusion_matrix.csv')
        ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(xticks_rotation=45,cmap='Blues'); plt.title(split); plt.tight_layout(); plt.savefig(OUT/f'{split}_confusion.png',dpi=150); plt.show()
else: print('No hay checkpoint entrenado; ejecuta la sección 4.')
history_path = OUT / "metrics_resumed.csv"

if not history_path.exists():
    history_path = OUT / "metrics_history.csv"

if history_path.exists():
    h = pd.read_csv(history_path)
    for metric in ['accuracy','loss','macro_f1']:
        plt.figure(figsize=(9,3));
        for col in ['train_'+metric,'val_'+metric]:
            if col in h: plt.plot(np.arange(1,len(h)+1),h[col],label=col)
        plt.legend(); plt.xlabel('Época global'); plt.title(metric); plt.tight_layout(); plt.savefig(OUT/f'history_{metric}.png',dpi=150); plt.show()

validation:   4%|▍         | 87/2128 [00:04<01:32, 21.98it/s]

In [24]:
@torch.no_grad()
def predict_image(path,checkpoint=CHECKPOINT):
    ck=torch.load(checkpoint,map_location=device,weights_only=True)
    assert ck['config']['classes']==CLASSES and ck['config']['mode']==MODE and ck['config']['fusion']==FUSION
    model.load_state_dict(ck['state_dict']); model.eval(); im=read_image(Path(path))
    logits,g=model(cnn_tf(im)[None].to(device),vit_tf(im)[None].to(device),True)
    p=logits.softmax(1)[0].cpu().numpy()
    return pd.Series(p,index=CLASSES).sort_values(ascending=False)
# display(predict_image(r'C:\ruta\rostro.jpg'))